# Quadern d'experimentació per obtindre el WER produït pel model **Whisper** amb el corpus **Europarl** (transcripció en castellà)

Realiztem totes les importacions que requerim per a la realització del experiment, i carreguem el model de whisper que utiliztarem (en aquest cas el model large-v3)

In [1]:
import whisper
import jiwer
from whisper.normalizers.basic import BasicTextNormalizer

from tqdm.notebook import tqdm
import pandas as pd

model = whisper.load_model("large-v3")

Carreguem la llista d'audios en castellà

In [2]:
audios = []

with open(r"..\Europarl-ST\es\en\test\Europarl-ST.v2.es.en.test.lst", "r", encoding="utf-8") as lista_audios:
    for linea in lista_audios:
        audios.append(str(linea).strip())

Realitzem la transcripció de tots els audios, i les guardem en _hypotheses_. Juntament, obrim i guardem les referències dels audios en _references_.

In [ ]:
hypotheses = []
references = []

for audio in tqdm(audios):
    with open(r"..\Europarl-ST\es\en\test\%s\transcription.tok" % audio, "r", encoding="utf-8") as referencia:
        references.append(referencia.read())

    hypotheses.append((model.transcribe(r"..\Europarl-ST\es\en\test\%s\audio_clip_diarization.m4a" % audio, language="Spanish"))['text'])

In [21]:
data = pd.DataFrame(dict(hypothesis=hypotheses, reference=references))
data

,hypothesis,reference
0,"Gracias, señor presidente. En primer lugar, q...","−\nSeñor Presidente , en primer lugar , quisie..."
1,"El Atlético de Madrid, los aficionados e incl...","El Atlético de Madrid , los aficionados e incl..."
2,"Gracias, señor presidente. Señor comisario, e...","Señor Presidente , señor Comisario , el terror..."
3,"Muchas gracias, señor presidente. Yo quiero a...","Señor Presidente , quiero agradecer a la Comis..."
4,"Gracias, presidente. Una gran mayoría de agri...","Señor Presidente , una gran mayoría de agricul..."
...,...,...
205,Yo también quiero felicitar al ponente y al r...,"Señor Presidente , yo también quiero felicitar..."
206,Me gusta el informe de la señora Thyssen porq...,"Señor Presidente , me gusta el informe de la s..."
207,"El 5,3% de la población activa trabaja en el ...","Señor Presidente , el 5,3 % de la población ac..."
208,"Presidente, debatimos hoy cuatro informes que...","Señor Presidente , debatimos hoy cuatro inform..."


Normalitzem el conjunt de dades (tant les hipòtesi com les referències)

In [22]:
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(text) for text in data["hypothesis"]]
data["reference_clean"] = [normalizer(text) for text in data["reference"]]
data

,hypothesis,reference,hypothesis_clean,reference_clean
0,"Gracias, señor presidente. En primer lugar, q...","−\nSeñor Presidente , en primer lugar , quisie...",gracias señor presidente en primer lugar quis...,señor presidente en primer lugar quisiera fel...
1,"El Atlético de Madrid, los aficionados e incl...","El Atlético de Madrid , los aficionados e incl...",el atlético de madrid los aficionados e inclu...,el atlético de madrid los aficionados e inclus...
2,"Gracias, señor presidente. Señor comisario, e...","Señor Presidente , señor Comisario , el terror...",gracias señor presidente señor comisario el t...,señor presidente señor comisario el terrorismo...
3,"Muchas gracias, señor presidente. Yo quiero a...","Señor Presidente , quiero agradecer a la Comis...",muchas gracias señor presidente yo quiero agr...,señor presidente quiero agradecer a la comisió...
4,"Gracias, presidente. Una gran mayoría de agri...","Señor Presidente , una gran mayoría de agricul...",gracias presidente una gran mayoría de agricu...,señor presidente una gran mayoría de agriculto...
...,...,...,...,...
205,Yo también quiero felicitar al ponente y al r...,"Señor Presidente , yo también quiero felicitar...",yo también quiero felicitar al ponente y al r...,señor presidente yo también quiero felicitar a...
206,Me gusta el informe de la señora Thyssen porq...,"Señor Presidente , me gusta el informe de la s...",me gusta el informe de la señora thyssen porq...,señor presidente me gusta el informe de la señ...
207,"El 5,3% de la población activa trabaja en el ...","Señor Presidente , el 5,3 % de la población ac...",el 5 3 de la población activa trabaja en el s...,señor presidente el 5 3 de la población activa...
208,"Presidente, debatimos hoy cuatro informes que...","Señor Presidente , debatimos hoy cuatro inform...",presidente debatimos hoy cuatro informes que ...,señor presidente debatimos hoy cuatro informes...


Calculem el **WER** de les trasncripcions de Whisper amb les referències

In [23]:
wer = jiwer.wer(list(data["reference_clean"]), list(data["hypothesis_clean"]))

print(f"WER: {wer * 100:.2f} %")

WER: 15.95 %


Observem que obtenim in WER de **15.95%**